# Test Notebook (using sample data)

## Import libraries and modules

In [1]:
import spectfbcalc_lib as sfc
from climtools import climtools_lib as ctl 
import output_lib as out

No DISPLAY variable set. Switching to agg backend


In [2]:
# Test libraries import
sfc.mytestfunction()

test!


In [3]:
ctl.datestamp()

'2026-09-17T08:33:53'

In [4]:
import sys
import os
import glob

import numpy as np
import xarray as xr

from matplotlib import pyplot as plt
import matplotlib.cbook as cbook

### OPTIONAL: launch workers to speed up the process (needs a SLURM scheduler)

In [5]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client

# Dask will automatically submit SLURM jobs for you
cluster = SLURMCluster(
    cores=4,
    memory="64GB",
    processes=4,
    walltime="01:00:00",
    job_extra_directives=[
        "--account=spitfabi",
        "--qos=np"
    ]
)

# Scale to desired number of workers
cluster.scale(jobs=4)  # This submits 4 SLURM jobs

# Connect client
client = Client(cluster)

In [6]:
print(client.dashboard_link)

http://10.100.192.102:8787/status


In [7]:
print(client)

<Client: 'tcp://10.100.192.102:36469' processes=0 threads=0, memory=0 B>


In [8]:
import dask.array as da
x = da.random.random((20000, 20000), chunks=(1000, 1000))
result = (x + x.T).mean().compute()
print(result)

# To check the status of the workers and the number of tasks executed, you can use the following code:
info = client.scheduler_info()['workers']
for addr, w in info.items():
    print(addr, "- tasks:", w.get('metrics', {}).get('task_counts', 'n/a'))

0.9999793305889636
tcp://10.100.133.12:37533 - tasks: {}
tcp://10.100.133.12:38853 - tasks: {}
tcp://10.100.133.12:43939 - tasks: {'released': 24, 'memory': 1}
tcp://10.100.133.12:44405 - tasks: {}
tcp://10.100.133.61:34409 - tasks: {'memory': 9, 'released': 2, 'executing': 1, 'ready': 1}


## Experiment setup

In [18]:
config_file='config_template.yaml'
config = sfc.load_config(config_file, variable_mapping_file = None)

Time range for climatology: all
Time range for experiment: all


In [19]:
config

{'kernels': {'HUANG': {'path_input': '../kernels/Huang/toa/',
   'filename_template': 'RRTMG_{}_toa_{}_highR.nc'},
  'SPECTRAL': {'path_input': '../kernels/spectral/toa/'}},
 'file_paths': {'reference_dataset': '../sample_data/reference/',
  'experiment_dataset': '../sample_data/experiment/',
  'pressure_data': '',
  'output': '../output/'},
 'control_type': 'PI',
 'exp_type': '4x',
 'exp_name': 'model_1',
 'anomaly_method': '',
 'save_pattern': False,
 'num_year_regr': 10,
 'num_running_years_trend': 25,
 'time_range_clim': None,
 'time_range_exp': None,
 'lon_range': {'start': '', 'end': ''},
 'lat_range': {'start': '', 'end': ''},
 'time_chunk': 12,
 'cart_out_exp': '../output//model_1/',
 'use_atm_mask': True,
 'pressure_path': '',
 'variable_mapping': None}

In [20]:
# Load the configuration file and preprocess the data
ker='HUANG' # or 'ERA5' or 'SPECTRAL'
control, experiment, kernel = sfc.preprocess_data(config = config, ker = ker)

Loading kernel: HUANG
Lat range to apply: {'start': '', 'end': ''}
Lon range to apply: {'start': '', 'end': ''}
no variable given. Using: "hus", "rlut", "rsdt", "rlutcs", "rsut", "rsutcs", "ta", "tas", "ts", "rsds", "rsus"

 -------> Loading control


FileNotFoundError: No files found for variable tas

In [ ]:
# OR: Load the control experiment object directly, without preprocessing the data
import yaml
with open('config_template.yaml', 'r') as f:
    config_dict = yaml.safe_load(f)

ker = 'HUANG'  # or 'ERA5' or 'SPECTRAL'
raw_variables = {"hus", "rlut", "rsdt", "rlutcs", "alb", "rsut", "rsutcs", "ta", "tas", "ts"}
control = sfc.Experiment('PI', config_dict['file_paths']['reference_dataset'], remap_dir = config_dict['file_paths']['output'] + f"remapped_{ker}/", raw_variables = raw_variables, variable_mapping = config_dict['variable_mapping'])

## Compute decomposed radiative anomalies

In [ ]:
# Compute anomalies one by one and save them in the specified folder
cart_out='./output/'
sfc.Rad_anomaly_planck_atm_lr(experiment,  kernel, cart_out)
sfc.Rad_anomaly_wv(experiment, control, kernel, cart_out)
sfc.Rad_anomaly_albedo(experiment, kernel, cart_out)
sfc.Rad_anomaly_planck_surf(experiment, kernel, cart_out)
sfc.Rad_anomaly_cloud(experiment, cart_out)

In [ ]:
# Compute anomalies all together and save them in the specified folder
sfc.calc_anoms(experiment, control, kernel, cart_out)

## Compute feedbacks

In [ ]:
# Compute single feedbacks and save them in the specified folder
name='albedo'
alb=sfc.single_feedback(name, experiment, kernel, cart_out)

In [ ]:
# Compute all feedbacks and save them in the specified folder
fb=sfc.calc_fb(experiment, control, kernel, cart_out)

In [ ]:
# Compute interannual feedbacks and save them in the specified folder
fb_interannual=sfc.calc_fb_interannual(experiment, control, kernel, cart_out)

## Save output and plot example

In [ ]:
out_path_txt='./output/results_fb.txt'
out.save_feedback_output(fb, out_path_txt)

In [ ]:
feedback_file=out_path_txt
out.plot_single_feedback_file(feedback_file)